# Compare Email and Paid Social

*Data Analysis in Python · Lesson 11 final exercise*

Your media team is setting next year's budget, and they want to know whether the customers Paid Social brings in are worth as much as Email's. Compare the customers both channels brought in from January to June 2024: how many ordered again, and which RFM segment they're in today.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

Add your cells between the cell that reads the data and the **Check your work** cell. If you're stuck on a step, open its hint under the **Check your work** cell.

1. Run the parameters cell and the cell that reads the data: the `customers` table and the `valid_orders` table, the orders before `CUTOFF` that weren't cancelled.
2. Create the `rfm` table, as in topic 11.3.'s exercise: recency, frequency, monetary value, the three scores and `segment` for every customer in the `valid_orders` table, as of `CUTOFF`.
3. Create the `h1_customers` table: the customers created from `COHORT_START` to before `COHORT_END`. Then create the `h1_segments` table: the `h1_customers` table with each customer's `frequency` and `segment` from the `rfm` table, for the customers who have them. A customer whose orders were all cancelled has no segment.
4. Create `repeat_rates`: for each `acquisition_channel`, the share of the `h1_segments` table's customers who ordered more than once.
5. Create the `segment_mix` table: one row for each channel in `CHANNELS` and one column per segment, with the share of the channel's customers in the `h1_segments` table in each segment. Each row adds up to 1.
6. Add a text cell for the media team: how the two channels' repeat rates compare, where their customers are today, what could explain the gap besides the channel, and what they should do with it.
7. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `rfm` table has 41,776 customers. The `h1_customers` table has 7,545 customers, and 7,265 of them are in the `h1_segments` table. `repeat_rates` has 9 channels: Email's is about `0.4158` and Paid Social's about `0.2688`. `segment_mix` has a row for Email and one for Paid Social: 7.6% of Email's customers are Champions, against 0.8% of Paid Social's.

## Parameters

In [ ]:
import numpy as np
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
CUTOFF = "2026-01-01"  # The timestamps are in UTC: the orders before it are the orders of 2024 and 2025.
COHORT_START, COHORT_END = "2024-01-01", "2024-07-01"
CHANNELS = ["Email", "Paid Social"]

In [ ]:
# The customers, and the orders of 2024 and 2025 that weren't cancelled. Run it, and leave it as it is.
customers = pd.read_csv(DATA_URL + "customers.csv", parse_dates=["created_at"])
orders = pd.read_csv(DATA_URL + "orders.csv", parse_dates=["created_at"])
orders = orders[orders["created_at"] < CUTOFF]
valid_orders = orders[orders["status"] != "Cancelled"]
len(customers), len(valid_orders)

In [ ]:
rfm = valid_orders.groupby("customer_id").agg(last_order=("created_at", "max"), frequency=("order_id", "count"),
                                              monetary=("subtotal", "sum"))
rfm["recency"] = (pd.Timestamp(CUTOFF) - rfm["last_order"]).dt.days
rfm["r_score"] = pd.qcut(rfm["recency"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["f_score"] = pd.cut(rfm["frequency"], bins=[0, 1, 2, 3, 5, np.inf], labels=[1, 2, 3, 4, 5]).astype(int)
rfm["m_score"] = pd.qcut(rfm["monetary"], 5, labels=[1, 2, 3, 4, 5]).astype(int)


def rfm_segment(r_score, f_score):
    """Return the RFM segment of a customer's recency and frequency scores."""
    if r_score >= 4 and f_score >= 4:
        return "Champions"
    if f_score >= 3:
        return "Loyal"
    if r_score >= 4 and f_score == 1:
        return "New"
    if r_score <= 2 and f_score >= 2:
        return "At risk"
    if r_score <= 2:
        return "Lost"
    return "Promising"


def row_segment(row):
    """Return the RFM segment of one row of the rfm table."""
    return rfm_segment(row["r_score"], row["f_score"])


rfm["segment"] = rfm.apply(row_segment, axis=1)
rfm["segment"].value_counts()

In [ ]:
h1_customers = customers[(customers["created_at"] >= COHORT_START) & (customers["created_at"] < COHORT_END)]
h1_segments = h1_customers.merge(rfm[["frequency", "segment"]], on="customer_id", how="inner", validate="one_to_one")
len(h1_customers), len(h1_segments)

In [ ]:
h1_segments["repeat"] = h1_segments["frequency"] > 1
repeat_rates = h1_segments.groupby("acquisition_channel")["repeat"].mean()
repeat_rates.sort_values(ascending=False).round(3), h1_segments["repeat"].mean()

In [ ]:
counts = pd.crosstab(h1_segments["acquisition_channel"], h1_segments["segment"]).loc[CHANNELS]
segment_mix = counts.div(counts.sum(axis=1), axis=0)
segment_mix.round(3), counts

**Email and Paid Social customers, acquired January to June 2024.** 736 customers made their first purchase through Email and 718 through Paid Social, not counting the few whose orders were all cancelled.

- **Email's customers came back far more often.** By the end of 2025, 41.6% of them had ordered more than once, 306 customers, against 26.9% of Paid Social's, 193. Across all channels, it's 34.7%.
- **Today, Email's customers are worth more.** On January 1, 2026, 7.6% of Email's customers are Champions and 11.4% Loyal, against 0.8% and 4.2% of Paid Social's. 72.7% of Paid Social's customers are Lost, against 58.3% of Email's.
- **Part of the gap is who they are.** Email's customers had subscribed to the newsletter before they bought, so they were already interested in the store. Paid Social reached people who hadn't heard of it. This comparison can't say how much of the gap the channel itself causes.
- **Next step.** Compare the channels on what a customer brings in over time, not on the cost of the first order: at the same cost per new customer, an Email customer is worth more. Send Email's 120 At risk customers a win-back offer now. For Paid Social, test whether a second-order offer in the first month lifts its repeat rate before cutting its budget.

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def number(value):
        return pd.api.types.is_number(value) and not isinstance(value, (pd.DataFrame, pd.Series))

    def near(value, *targets, tolerance=0.0001):
        return number(value) and any(abs(value - target) < tolerance for target in targets)

    rfm = found.get("rfm")
    if not isinstance(rfm, pd.DataFrame) or "segment" not in rfm.columns:
        results.append((False, "rfm should be a DataFrame with a segment column: topic 11.3.'s table, for every customer in the valid_orders table."))
    elif len(rfm) == 43509:
        results.append((False, "The rfm table has 43,509 customers: it counts cancelled orders. Use the valid_orders table."))
    elif len(rfm) == 7265:
        results.append((False, "The rfm table has only the cohort's 7,265 customers. A score is relative to the customers you scored, so score all 41,776 customers in the valid_orders table, then look at the cohort."))
    elif len(rfm) != 41776:
        results.append((False, f"The rfm table should have the 41,776 customers in the valid_orders table. It has {len(rfm):,}."))
    elif rfm["segment"].value_counts().to_dict() != {"Lost": 14069, "Promising": 11437, "New": 10650, "At risk": 2101, "Loyal": 2071, "Champions": 1448}:
        results.append((False, "The rfm table's segments should match topic 11.3.'s exercise: 14,069 Lost customers and 1,448 Champions. Check the scores and the order of rfm_segment()'s rules."))
    else:
        results.append((True, "Every customer's RFM segment as of January 1, 2026, as in topic 11.3.'s exercise."))

    h1 = found.get("h1_customers")
    if not isinstance(h1, pd.DataFrame):
        results.append((False, "h1_customers should be a DataFrame: the customers created from COHORT_START to before COHORT_END."))
    elif len(h1) == 8801:
        results.append((False, "The h1_customers table has 8,801 customers: it starts each customer at their first order in the file, so 2023 customers who came back in 2024 count as new. Use the customers table's created_at."))
    elif len(h1) == 53272:
        results.append((False, "The h1_customers table has all 53,272 customers. Keep the ones created from COHORT_START to before COHORT_END, so every customer has had the same time to come back."))
    elif len(h1) != 7545:
        results.append((False, f"The h1_customers table should have the 7,545 customers created from COHORT_START to before COHORT_END. It has {len(h1):,}."))
    else:
        results.append((True, "7,545 customers created from January to June 2024: 756 through Email and 757 through Paid Social."))

    segments = found.get("h1_segments")
    if not isinstance(segments, pd.DataFrame) or [column for column in ["acquisition_channel", "frequency", "segment"] if column not in segments.columns]:
        results.append((False, "h1_segments should be a DataFrame: the h1_customers table, with frequency and segment from the rfm table."))
    elif len(segments) == 7545:
        results.append((False, "The h1_segments table has all 7,545 cohort customers, including 280 whose orders were all cancelled and who have no segment. Use an inner join: how=\"inner\"."))
    elif len(segments) != 7265:
        results.append((False, f"The h1_segments table should have the 7,265 cohort customers who are in the rfm table. It has {len(segments):,}."))
    else:
        results.append((True, "7,265 cohort customers with a segment: 736 through Email and 718 through Paid Social."))

    rates = found.get("repeat_rates")
    if not isinstance(rates, pd.Series) or "Email" not in rates.index or "Paid Social" not in rates.index:
        results.append((False, "repeat_rates should be a Series with one share per acquisition_channel."))
    elif len(rates) != 9:
        results.append((False, f"repeat_rates should have the 9 acquisition channels. It has {len(rates)}."))
    elif near(rates["Email"], 1.0) and near(rates["Paid Social"], 1.0):
        results.append((False, "repeat_rates is 1 for every channel: it counts the customers with at least one order, and every customer in the h1_segments table has one. Count the ones with more than one: frequency > 1."))
    elif not near(rates["Email"], 0.415761) or not near(rates["Paid Social"], 0.268802):
        results.append((False, f"repeat_rates should be about 0.4158 for Email and 0.2688 for Paid Social: the mean of frequency > 1 per channel, in the h1_segments table. They are {rates['Email']!r} and {rates['Paid Social']!r}."))
    else:
        results.append((True, f"By the end of 2025, {rates['Email']:.1%} of Email's customers had ordered more than once, and {rates['Paid Social']:.1%} of Paid Social's."))

    mix = found.get("segment_mix")
    if not isinstance(mix, pd.DataFrame) or "Email" not in mix.index or "Paid Social" not in mix.index:
        results.append((False, "segment_mix should be a DataFrame with a row for Email and one for Paid Social."))
    elif "Champions" not in mix.columns or "Lost" not in mix.columns:
        results.append((False, "segment_mix should have one column per segment, such as Champions and Lost."))
    elif near(mix.loc["Email", "Champions"], 56):
        results.append((False, "segment_mix has the number of customers in each segment. Divide each row by its total, so each row adds up to 1: counts.div(counts.sum(axis=1), axis=0)."))
    elif near(mix.loc["Email", "Champions"], 0.129076):
        results.append((False, "segment_mix shows 12.9% of Email's customers as Champions: the scores were calculated among the cohort's customers only. Score every customer first, then look at the cohort."))
    elif not near(mix.loc["Email", "Champions"], 0.076087) or not near(mix.loc["Paid Social", "Lost"], 0.727019):
        results.append((False, f"segment_mix should have the share of each channel's customers in the h1_segments table in each segment, with each row adding up to 1: 7.6% of Email's are Champions, and 72.7% of Paid Social's are Lost. It has {mix.loc['Email', 'Champions']!r} and {mix.loc['Paid Social', 'Lost']!r}."))
    else:
        results.append((True, "7.6% of Email's customers are Champions and 11.4% Loyal, against 0.8% and 4.2% of Paid Social's. 72.7% of Paid Social's are Lost."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run. Read your text cell again: it's what the media team will budget with.")


check_your_work()

## Hints

Stuck on a step? Open its hint. Each one says where the course showed it, and gives the shape of the code, with `...` where your part goes. Try the step from the hint before you open the solution.

<details>
<summary><b>Step 2: the RFM segments</b></summary>

Topic 11.3.'s exercise, on every customer: a score is relative to the customers you scored, so score them all before you look at the cohort.

~~~python
rfm = valid_orders.groupby("customer_id").agg(last_order=..., frequency=..., monetary=...)
rfm["recency"] = (pd.Timestamp(CUTOFF) - rfm["last_order"]).dt.days
rfm["r_score"] = pd.qcut(rfm["recency"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["f_score"] = pd.cut(rfm["frequency"], bins=[0, 1, 2, 3, 5, np.inf], labels=[1, 2, 3, 4, 5]).astype(int)
rfm["m_score"] = ...


def rfm_segment(r_score, f_score):
    ...


def row_segment(row):
    ...


rfm["segment"] = rfm.apply(row_segment, axis=1)
~~~

</details>

<details>
<summary><b>Step 3: the cohort and its segments</b></summary>

Topic 11.2.: start from the customer record, not from the first order in the file, which for a 2023 customer isn't their first. Then join the segments as in topic 11.1.: `on="customer_id"` works when it's the name of the `rfm` table's index. An inner join keeps the customers who have a segment.

~~~python
h1_customers = customers[(customers["created_at"] >= ...) & (customers["created_at"] < ...)]
h1_segments = h1_customers.merge(rfm[["frequency", "segment"]], on="customer_id", how=..., validate="one_to_one")
len(h1_customers), len(h1_segments)
~~~

</details>

<details>
<summary><b>Step 4: the repeat rates</b></summary>

The share of `True` per channel is a mean (topic 8.2.).

~~~python
h1_segments["repeat"] = h1_segments["frequency"] > ...
repeat_rates = h1_segments.groupby(...)["repeat"].mean()
~~~

</details>

<details>
<summary><b>Step 5: the segment mix</b></summary>

Count the customers in each channel and segment with topic 9.3.'s `pd.crosstab()`, keep the two channels' rows, and divide each row by its total with topic 11.2.'s `div(axis=0)`.

~~~python
counts = pd.crosstab(h1_segments[...], h1_segments[...]).loc[CHANNELS]
segment_mix = counts.div(counts.sum(axis=1), axis=0)
~~~

</details>

<details>
<summary><b>Step 6: the text cell</b></summary>

Choose **Insert → Text cell**. Start with the repeat rates, with the customer counts behind them. Then compare the segments that matter for revenue: Champions, Loyal and At risk. Before you recommend anything, say what else could explain the gap: Email's customers had subscribed to the newsletter before they bought. End with how the media team should compare the channels' costs.

</details>

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
rfm = valid_orders.groupby("customer_id").agg(last_order=("created_at", "max"), frequency=("order_id", "count"),
                                              monetary=("subtotal", "sum"))
rfm["recency"] = (pd.Timestamp(CUTOFF) - rfm["last_order"]).dt.days
rfm["r_score"] = pd.qcut(rfm["recency"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["f_score"] = pd.cut(rfm["frequency"], bins=[0, 1, 2, 3, 5, np.inf], labels=[1, 2, 3, 4, 5]).astype(int)
rfm["m_score"] = pd.qcut(rfm["monetary"], 5, labels=[1, 2, 3, 4, 5]).astype(int)


def rfm_segment(r_score, f_score):
    """Return the RFM segment of a customer's recency and frequency scores."""
    if r_score >= 4 and f_score >= 4:
        return "Champions"
    if f_score >= 3:
        return "Loyal"
    if r_score >= 4 and f_score == 1:
        return "New"
    if r_score <= 2 and f_score >= 2:
        return "At risk"
    if r_score <= 2:
        return "Lost"
    return "Promising"


def row_segment(row):
    """Return the RFM segment of one row of the rfm table."""
    return rfm_segment(row["r_score"], row["f_score"])


rfm["segment"] = rfm.apply(row_segment, axis=1)
rfm["segment"].value_counts()
```

```python
h1_customers = customers[(customers["created_at"] >= COHORT_START) & (customers["created_at"] < COHORT_END)]
h1_segments = h1_customers.merge(rfm[["frequency", "segment"]], on="customer_id", how="inner", validate="one_to_one")
len(h1_customers), len(h1_segments)
```

```python
h1_segments["repeat"] = h1_segments["frequency"] > 1
repeat_rates = h1_segments.groupby("acquisition_channel")["repeat"].mean()
repeat_rates.sort_values(ascending=False).round(3), h1_segments["repeat"].mean()
```

```python
counts = pd.crosstab(h1_segments["acquisition_channel"], h1_segments["segment"]).loc[CHANNELS]
segment_mix = counts.div(counts.sum(axis=1), axis=0)
segment_mix.round(3), counts
```

```markdown
**Email and Paid Social customers, acquired January to June 2024.** 736 customers made their first purchase through Email and 718 through Paid Social, not counting the few whose orders were all cancelled.

- **Email's customers came back far more often.** By the end of 2025, 41.6% of them had ordered more than once, 306 customers, against 26.9% of Paid Social's, 193. Across all channels, it's 34.7%.
- **Today, Email's customers are worth more.** On January 1, 2026, 7.6% of Email's customers are Champions and 11.4% Loyal, against 0.8% and 4.2% of Paid Social's. 72.7% of Paid Social's customers are Lost, against 58.3% of Email's.
- **Part of the gap is who they are.** Email's customers had subscribed to the newsletter before they bought, so they were already interested in the store. Paid Social reached people who hadn't heard of it. This comparison can't say how much of the gap the channel itself causes.
- **Next step.** Compare the channels on what a customer brings in over time, not on the cost of the first order: at the same cost per new customer, an Email customer is worth more. Send Email's 120 At risk customers a win-back offer now. For Paid Social, test whether a second-order offer in the first month lifts its repeat rate before cutting its budget.
```

</details>